# TMDWF CS-Kernel Extraction Template

This notebook is a standalone wrapper around the repository's downstream TMDWF Collins-Soper kernel extraction workflow.
It reads already-generated TMDWF Fourier outputs, preserves the legacy type-2 bootstrap extraction logic, and writes summary bands, diagnostics, and bootstrap samples without rerunning the original TMDWF fit or Fourier step.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tmdwf_cs_kernel_input_text,
    run_tmdwf_cs_kernel_from_notebook,
    validate_tmdwf_cs_kernel_notebook_config,
)


## User Inputs

These fields describe a downstream CS-kernel extraction job.
The workflow expects repository-native TMDWF Fourier outputs to already exist under `input_root`.


In [ ]:
workflow_config = {
    # Data settings
    "title_pattern": "demo_tmdwf_cs_kernel",
    "input_root": str(REPO_ROOT / "examples" / "outputs" / "tmdwf_fourier_notebook"),
    "ns": 64,
    "lattice_spacing_fm": 0.076,
    "gmlist": ["T5"],
    "etalist": ["eta0"],
    "component": "real",
    "nstates": 1,
    "normalization_mode": "raw",
    "mu": 2.0,

    # CS-kernel extraction settings
    "scheme": "CG",
    "extraction_type": "type2",
    "pair_mode": "all",
    "reference_p1": None,
    "kernel_labels": ["LO", "NLO", "NLL"],
    "bTlist": [0, 2, 4],
    "pzlist": [2, 3, 4, 5, 6],
    "x_window": [0.2, 0.8],

    # Output settings
    "plot": True,
    "results_dir": str(REPO_ROOT / "examples" / "outputs" / "tmdwf_cs_kernel"),
}
workflow_config


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

- `input_root`: Root directory containing existing TMDWF Fourier outputs. The workflow automatically resolves the correct Fourier tables and bootstrap sample files from the usual repository naming convention.
- `title_pattern`: Same per-`pz` title pattern used upstream by the TMDWF fit/Fourier workflows, for example `l64c64a076_m140_fit_pz*`.
- `ns`, `lattice_spacing_fm`: Ensemble metadata used to convert lattice momentum integers into physical momenta via `2*pi/(Ns*a*fmGeV)`.
- `gmlist`, `etalist`: Select which operator/insertion channels to read from the existing Fourier outputs.
- `component`, `nstates`: Select which Fourier output family to consume.
- `normalization_mode`: One of `raw`, `mode1`, `mode2`, or `mode3`. This matches the upstream Fourier output mode and is recorded explicitly in the CS-kernel outputs.
- `mu`: Perturbative matching scale in GeV passed to the legacy `CS_Dgamma` correction object.
- `scheme`: Matching scheme selector. The current repository implementation supports only `"CG"` for the type-2 TMDWF workflow, and will raise a clear validation error otherwise.
- `extraction_type`: Keep this at `"type2"` for the legacy qTMDWF CS-kernel method. The field is explicit so later extensions can add other extraction strategies without changing the interface.
- `pair_mode`: Momentum-combination mode. `all` means one output group per `bT`, using a shared `P1` and all other requested momenta as `P2`. `adjacent` means one output group per neighboring pair, e.g. `5-6`, `6-7`, `7-8`. `fixed_p1` means one output group per `P2` using a shared `P1`, and it only writes the pairwise breakdown plot.
- `reference_p1`: Optional shared P1 momentum. When omitted, the workflow uses `pzlist[0]`. When provided, it is used by `all` and `fixed_p1`. `fixed_p1` filenames are tagged with `fixedp1_...` so they do not collide with `all`-mode files.
- `kernel_labels`: One or more perturbative labels to run in batch. The current mapping follows the legacy scripts: `LO -> 0`, `NLO/NLL -> 1`, `NNLO/NNLL -> 2`.
- `bTlist` or `bTrange`: Which transverse separations to process. The workflow loops over every requested `bT`.
- `pzlist` or `pzrange`: Which lattice momentum integers to include. How they are grouped into `P1/P2` combinations is controlled explicitly by `pair_mode`.
- `x_window`: Fit window in `x`. The default `[0.2, 0.8]` reproduces the legacy extraction window used by the old multi-`Pz` script.
- `plot`: Whether to also write a quick summary PDF for each `(kernel_label, bT, pair-group)` band. When `pair_mode` is `adjacent` or `fixed_p1`, the workflow writes one automatic pairwise breakdown plot showing the data log-ratio term, the matching correction, and the total estimator.
- `results_dir`: Output root for the generated summary files, band tables, bootstrap samples, diagnostics, and optional plots.

Expected input data shape:

- The workflow reads the repository-native Fourier sample table format: one row per `(sample_id, x)` with a `q_sample` column.
- Internally those long-form rows are regrouped into a bootstrap matrix with shape `(n_samples, n_x)` before the CS estimator and constant fit are applied.

What the workflow writes for each `(kernel_label, bT, pair-group)`:

- `*_summary.txt`: metadata/provenance snapshot
- `tables/*_band.txt`: `x` plus 16/50/84 CS-kernel quantiles
- `samples/*_samples.txt`: one row per `(x, sample_id)` bootstrap value
- `diagnostics/*_diagnostics.txt`: `chi2/dof` quantiles vs `x`
- `plots/*_band.pdf`: optional quick-look plot when `plot true` and `pair_mode` is `all`
- `plots/*_pairwise_breakdown.pdf`: automatic breakdown plot when `plot true` and `pair_mode` is `adjacent` or `fixed_p1`


## Validate Config

This uses the same parser as the CLI workflow, so it is a good way to confirm the text rendering and defaults before running.


In [ ]:
validated = validate_tmdwf_cs_kernel_notebook_config(workflow_config)
validated


## Render Input Preview

This is the plain-text control file that the notebook helper materializes behind the scenes.


In [ ]:
input_preview = render_tmdwf_cs_kernel_input_text(workflow_config)
print(input_preview)


## Run Workflow

This launches the repository-native CS-kernel extraction workflow and prints the generated artifacts.


In [ ]:
outputs = run_tmdwf_cs_kernel_from_notebook(workflow_config)
for output in outputs:
    print(output)


## Config Snapshot

This is useful to keep alongside saved results.


In [ ]:
print(pretty_print_config(workflow_config))
